# NYC Airbnb Market Analysis

Import, clean, and analyze listing data from three source files (CSV, Excel, TSV).

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
DATA_DIR = Path("data")

## 1. Ingest

Load the three raw files and inspect shape, dtypes, and missing values.

In [2]:
price_df = pd.read_csv(DATA_DIR / "airbnb_price.csv", dtype={"listing_id": str})
room_df = pd.read_excel(DATA_DIR / "airbnb_room_type.xlsx", dtype={"listing_id": str})
review_df = pd.read_csv(DATA_DIR / "airbnb_last_review.tsv", sep="\t", dtype={"listing_id": str})

print("Price CSV:", price_df.shape)
print(price_df.dtypes)
print("Missing values:\n", price_df.isna().sum())
print()
print("Room type Excel:", room_df.shape)
print(room_df.dtypes)
print("Missing values:\n", room_df.isna().sum())
print()
print("Last review TSV:", review_df.shape)
print(review_df.dtypes)
print("Missing values:\n", review_df.isna().sum())

display(price_df.head(2))
display(room_df.head(2))
display(review_df.head(2))

Price CSV: (30234, 3)
listing_id     object
price          object
nbhood_full    object
dtype: object
Missing values:
 listing_id        0
price          9903
nbhood_full       0
dtype: int64

Room type Excel: (30234, 3)
listing_id     object
description    object
room_type      object
dtype: object
Missing values:
 listing_id     0
description    1
room_type      0
dtype: int64

Last review TSV: (30234, 3)
listing_id     object
host_name      object
last_review    object
dtype: object
Missing values:
 listing_id        0
host_name       563
last_review    8309
dtype: int64


,listing_id,price,nbhood_full
0,3581652,NaN,"Manhattan, Morningside Heights"
1,3585309,NaN,"Queens, Flushing"


,listing_id,description,room_type
0,3581652,Duplex in SoHa,Entire Home/Apt
1,3585309,Beautiful home away from home!,Entire home/apt


,listing_id,host_name,last_review
0,3581652,Sebastien,July 29 2016
1,3585309,Freddy,December 22 2018


## 2. Combine

Merge all three sources on `listing_id` into a single DataFrame.

In [3]:
df = price_df.merge(room_df, on="listing_id", how="inner")
df = df.merge(review_df, on="listing_id", how="inner")

print(f"Merged shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Duplicate listing_id rows: {df['listing_id'].duplicated().sum()}")
df.head()

Merged shape: (30234, 7)
Columns: ['listing_id', 'price', 'nbhood_full', 'description', 'room_type', 'host_name', 'last_review']
Duplicate listing_id rows: 0


,listing_id,price,nbhood_full,description,room_type,host_name,last_review
0,3581652,NaN,"Manhattan, Morningside Heights",Duplex in SoHa,Entire Home/Apt,Sebastien,July 29 2016
1,3585309,NaN,"Queens, Flushing",Beautiful home away from home!,Entire home/apt,Freddy,December 22 2018
2,3585974,NaN,"Manhattan, Lower East Side",1BD in Lower East Side,entire home/apt,Elisa,May 16 2018
3,3591107,NaN,"Brooklyn, Flatbush",Gorgeous Bedroom by the Park,PRIVATE ROOM,Lanny,January 1 2019
4,3593821,NaN,"Brooklyn, Williamsburg",Bedroom in Williamsburg w/ Terrace,PRIVATE ROOM,Eric,July 26 2015


## 3. Clean strings

Strip `" dollars"` from price and convert to numeric, normalize `room_type` casing to a category, and split `nbhood_full` into `borough` and `neighbourhood`.

In [4]:
# Price: "225 dollars" -> 225
df["price"] = (
    df["price"]
    .astype(str)
    .str.replace(" dollars", "", regex=False)
    .str.strip()
)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

# Room type: inconsistent casing -> canonical labels + category
room_type_map = {
    "entire home/apt": "Entire home/apt",
    "private room": "Private room",
    "shared room": "Shared room",
    "hotel room": "Hotel room",
}
df["room_type"] = (
    df["room_type"]
    .str.strip()
    .str.lower()
    .map(room_type_map)
    .astype("category")
)

# Neighbourhood: "Manhattan, Midtown" -> borough + neighbourhood
nbhood_parts = df["nbhood_full"].str.split(",", n=1, expand=True)
df["borough"] = nbhood_parts[0].str.strip()
df["neighbourhood"] = nbhood_parts[1].str.strip()

print("Price dtype:", df["price"].dtype)
print("Price non-null:", df["price"].notna().sum(), "| missing:", df["price"].isna().sum())
print("Price sample:", df["price"].dropna().head(3).tolist())
print()
print("Room type categories:", list(df["room_type"].cat.categories))
print(df["room_type"].value_counts())
print()
print("Borough value counts:")
print(df["borough"].value_counts())
df[["listing_id", "price", "room_type", "borough", "neighbourhood"]].head()

Price dtype: float64
Price non-null: 20331 | missing: 9903


Price sample: [149.0, 105.0, 180.0]

Room type categories: ['Entire home/apt', 'Hotel room', 'Private room', 'Shared room']
room_type
Entire home/apt    16693
Private room       12867
Hotel room           486
Shared room          188
Name: count, dtype: int64

Borough value counts:
borough
Manhattan        13872
Brooklyn         10374
Queens            4678
Bronx              987
Staten Island      323
Name: count, dtype: int64


,listing_id,price,room_type,borough,neighbourhood
0,3581652,NaN,Entire home/apt,Manhattan,Morningside Heights
1,3585309,NaN,Entire home/apt,Queens,Flushing
2,3585974,NaN,Entire home/apt,Manhattan,Lower East Side
3,3591107,NaN,Private room,Brooklyn,Flatbush
4,3593821,NaN,Private room,Brooklyn,Williamsburg


## 4. Format dates

Parse `last_review` text (e.g. `"August 3 2026"`) into datetime and report the earliest and latest review dates.

In [5]:
df["last_review"] = pd.to_datetime(
    df["last_review"], format="%B %d %Y", errors="coerce"
)

earliest_review = df["last_review"].min()
latest_review = df["last_review"].max()

print("last_review dtype:", df["last_review"].dtype)
print("Parsed reviews:", df["last_review"].notna().sum(), "| missing:", df["last_review"].isna().sum())
print(f"Earliest review: {earliest_review.date()}")
print(f"Latest review:   {latest_review.date()}")
df[["listing_id", "host_name", "last_review"]].head()

last_review dtype: datetime64[ns]
Parsed reviews: 21925 | missing: 8309
Earliest review: 2011-05-12
Latest review:   2026-08-20


,listing_id,host_name,last_review
0,3581652,Sebastien,2016-07-29
1,3585309,Freddy,2018-12-22
2,3585974,Elisa,2018-05-16
3,3591107,Lanny,2019-01-01
4,3593821,Eric,2015-07-26
